# 🚀 Representar texto: BOW, TF-IDF y Word2Vec, en la misma tarea
*Curso de NLP · 🚀 Llévalo a Colab · [MiAtlas.cl](https://miatlas.cl) · Licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es)*

**Objetivo:** convertir texto en números de tres maneras y comparar cuál clasifica mejor **emociones** en tweets.

En el curso de NLP construiste la bolsa de palabras, TF-IDF y Word2Vec desde cero. Aquí los usas con bibliotecas,
sobre 16.000 tweets, y al final entrenas una red en **GPU** sobre los *embeddings*.

**Los datos:** *emotion* (`dair-ai/emotion` en Hugging Face): tweets en inglés etiquetados con seis emociones
(tristeza, alegría, amor, enojo, miedo y sorpresa).

In [ ]:
# Preparación: ¿Colab?, ¿GPU? y modo de prueba
import os
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
PRUEBA_RAPIDA = os.environ.get("MIATLAS_PRUEBA") == "1"   # solo para la prueba automática de MiAtlas.cl

import torch
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
print("Colab:", EN_COLAB, "| dispositivo:", DISPOSITIVO)
if DISPOSITIVO == "cpu":
    print("⚠ Sin GPU. En Colab: Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU.")

In [ ]:
%pip install -q datasets nltk gensim

In [ ]:
from datasets import load_dataset
import pandas as pd

emociones = load_dataset("dair-ai/emotion")
NOMBRES = emociones["train"].features["label"].names
df_train = emociones["train"].to_pandas()
df_test = emociones["test"].to_pandas()
if PRUEBA_RAPIDA:
    df_train, df_test = df_train.sample(3000, random_state=0), df_test.sample(600, random_state=0)
print(NOMBRES)
print(len(df_train), "tweets para entrenar y", len(df_test), "para probar")
df_train.head()

In [ ]:
df_train["label"].map(dict(enumerate(NOMBRES))).value_counts().plot(kind="barh", title="Tweets por emoción");

## 1. Normalizar: *stemming* frente a lematización

- **Stemming** corta la palabra a su raíz con reglas («studies» → «studi»): rápido, pero deja raíces que no son
  palabras.
- **Lematización** busca la forma de diccionario («studies» → «study»): más lenta, más precisa.

In [ ]:
import nltk
from nltk.stem import SnowballStemmer, WordNetLemmatizer
nltk.download("wordnet", quiet=True)

raiz, lema = SnowballStemmer("english"), WordNetLemmatizer()
for palabra in ["studies", "running", "better", "feelings", "happily"]:
    print(f"{palabra:10} stem: {raiz.stem(palabra):8} lema: {lema.lemmatize(palabra, pos='v')}")

## 2. Bolsa de palabras (BOW) y TF-IDF

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

resultados = {}
for nombre, vectorizador in [("BOW", CountVectorizer(min_df=2)), ("TF-IDF", TfidfVectorizer(min_df=2, sublinear_tf=True))]:
    X_train = vectorizador.fit_transform(df_train["text"])
    X_test = vectorizador.transform(df_test["text"])
    modelo = LogisticRegression(max_iter=2000).fit(X_train, df_train["label"])
    pred = modelo.predict(X_test)
    resultados[nombre] = f1_score(df_test["label"], pred, average="macro")
    print(f"{nombre:7} vocabulario {X_train.shape[1]:6} · F1 macro {resultados[nombre]:.3f}")

## 3. Word2Vec: el significado como vector

Word2Vec aprende un vector por palabra a partir de sus vecinas. Para clasificar un tweet, promediamos los vectores de
sus palabras.

In [ ]:
import numpy as np
from gensim.models import Word2Vec

tokens_train = [t.lower().split() for t in df_train["text"]]
tokens_test = [t.lower().split() for t in df_test["text"]]
w2v = Word2Vec(tokens_train, vector_size=100, window=5, min_count=2, workers=2, seed=0, epochs=10)
print("Parecidas a 'happy':", [p for p, _ in w2v.wv.most_similar("happy", topn=5)])

def promedio(tokens):
    vectores = [w2v.wv[p] for p in tokens if p in w2v.wv]
    return np.mean(vectores, axis=0) if vectores else np.zeros(w2v.vector_size)

X_train_w2v = np.array([promedio(t) for t in tokens_train])
X_test_w2v = np.array([promedio(t) for t in tokens_test])
pred = LogisticRegression(max_iter=2000).fit(X_train_w2v, df_train["label"]).predict(X_test_w2v)
resultados["Word2Vec (promedio)"] = f1_score(df_test["label"], pred, average="macro")
print(f"Word2Vec · F1 macro {resultados['Word2Vec (promedio)']:.3f}")

> **¿Por qué Word2Vec promediado puede salir peor que TF-IDF?** Al promediar se pierden las palabras clave: «not
> happy» y «happy» quedan casi iguales. Los *embeddings* brillan cuando una red aprende a combinarlos, como en la
> sección 4.

## 4. Una red con *embeddings*, en GPU (Keras)

In [ ]:
try:
    import tensorflow as tf
except ImportError:
    tf = None
    print("TensorFlow no está instalado aquí (en Colab viene incluido): se omite esta sección.")

if tf is not None:
    print("GPU para TensorFlow:", tf.config.list_physical_devices("GPU") or "ninguna")
    capa_texto = tf.keras.layers.TextVectorization(max_tokens=20000, output_sequence_length=50)
    capa_texto.adapt(df_train["text"].values)
    red = tf.keras.Sequential([
        capa_texto,
        tf.keras.layers.Embedding(20000, 64),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(len(NOMBRES), activation="softmax"),
    ])
    red.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    red.fit(tf.constant(df_train["text"].values), df_train["label"].values,
            epochs=1 if PRUEBA_RAPIDA else 5, batch_size=64, verbose=2)
    pred = red.predict(tf.constant(df_test["text"].values), verbose=0).argmax(1)
    resultados["Red LSTM (Keras)"] = f1_score(df_test["label"], pred, average="macro")
    print(f"Red LSTM · F1 macro {resultados['Red LSTM (Keras)']:.3f}")

In [ ]:
pd.Series(resultados, name="F1 macro").sort_values().round(3)

## Pruébalo tú

Mejora TF-IDF: prueba `ngram_range=(1, 2)` (pares de palabras) y guarda el F1 macro en `f1_bigramas`. ¿Ayudan los
pares como «not happy»?

In [ ]:
f1_bigramas = None   # ← tu código

In [ ]:
# Comprobación
if f1_bigramas is None:
    print("Aún no lo resuelves.")
else:
    print(f"✅ TF-IDF con pares de palabras: {f1_bigramas:.3f} (antes: {resultados['TF-IDF']:.3f})")

## Resumen

- **BOW** cuenta palabras; **TF-IDF** baja el peso de las muy comunes; **Word2Vec** captura el significado.
- Promediar *embeddings* pierde información; una **red** que los combina la recupera, y la GPU la entrena rápido.

**Fuentes:** [scikit-learn · texto](https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction) · [gensim · Word2Vec](https://radimrehurek.com/gensim/models/word2vec.html) · [NLTK · stem](https://www.nltk.org/api/nltk.stem.html) · [Keras · TextVectorization](https://keras.io/api/layers/preprocessing_layers/text/text_vectorization/) · [dair-ai/emotion](https://huggingface.co/datasets/dair-ai/emotion)